# BBBP Molecular Property Prediction: Experiment 2
### Traditional Machine Learning and GCN Baselines Comparison under Bemis-Murcko Scaffold Split

This notebook implements the complete pipeline for **Experiment 2** as outlined in the experimental plan. It compares our proposed dual-channel **MolFormer + MPNN Gated Fusion** model against standard machine learning and graph neural network baselines under a rigorous **Bemis-Murcko Scaffold Split** of 80% train, 10% validation, and 10% test. 

## 🌟 Core Features & Academic Upgrades
- **⚖️ Bemis-Murcko Scaffold Partitioning**: Eliminates data leakage, evaluating generalization on structurally novel drug scaffolds.
- **📦 Multi-Baseline Evaluation**: Runs RF, SVM, GBDT (XGBoost/sklearn), and custom PyTorch GCN in one unified notebook.
- **🎯 PyTorch GCN from Scratch**: Implements a custom 3-layer Graph Convolutional Network without external dependencies (no PyG/DGL).
- **📊 Publication-Quality Plots (300 DPI)**:
  - **ROC Curves**: Side-by-side comparison of test set ROC curves.
  - **Precision-Recall (PR) Curves**: Critical for assessing highly skewed molecular datasets.
  - **Grouped Metrics Bar Chart**: Visually compares ROC-AUC, PR-AUC, MCC, F1, and Accuracy across all models.

## 🛠️ Step 0: Environment Setup & Library Installation
Install RDKit, Hugging Face Transformers, and ensure other required modeling and visualization libraries are available.

In [1]:
# Install required dependencies
!pip install -q rdkit matplotlib seaborn xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 56.5 MB/s eta 0:00:00


In [2]:
!pip install "transformers<4.35.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.5/121.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.0/295.0 kB 19.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.10.1
    Uninstalling huggingface_hub-1.10.1:
      Successfully uninstalled huggingface_hub-1.10.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the fol

## 🛠️ Step 1: Library Imports & Configuration Setup
Set random seeds, configure Matplotlib for publication aesthetics, and check device availability (CPU/GPU).

In [3]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, matthews_corrcoef, accuracy_score, f1_score, roc_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from collections import defaultdict

# RDKit imports
from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import AllChem, Descriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

# Hugging Face Transformers
from transformers import AutoModel, AutoTokenizer

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.*")

import random
SEED = 42
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

MOLFORMER_NAME = "/kaggle/input/models/konstantinboyko/molformer-xl-both-10pct-ibm-research/transformers/default/1"
DATASET_PATH = "/kaggle/input/datasets/shixinguo/bbbp-combined/BBBP_combined.csv"
BATCH_SIZE = 32
EPOCHS = 40
LR = 5e-4
MPNN_DIM = 64
MOLFORMER_DIM = 768

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device Info] Active device for training: {device}")

# Output directories
os.makedirs("results", exist_ok=True)
os.makedirs("results/plots", exist_ok=True)

# Academic Plotting Aesthetics Setup
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['figure.titlesize'] = 15

[Device Info] Active device for training: cuda


## 📂 Step 2: Dataset Loading and Auto-Discovery
Load target variables and molecular structures from BBBP combined CSV dataset.

In [4]:
# Load target columns: indices, SMILES string, and binary label
df = pd.read_csv(DATASET_PATH, usecols=[1, 2, 3])
df = df.rename(columns={'label': 'permeability_target'})
print(f"✅ Dataset successfully loaded. Total molecules: {len(df)}")
df.head()

✅ Dataset successfully loaded. Total molecules: 3120


,name,permeability_target,smiles
0,SB204459,0,c1(ccc(c(c1)Cl)Cl)CC(N1[C@H](C[N@@]2C[C@@H](CC...
1,oximonam,0,CO\N=C(C(=O)NC1[C@H](C)N(OCC(O)=O)C1=O)\c2csc(...
2,aztreonam,0,C[C@H]1[C@H](NC(=O)C(=N/OC(C)(C)C(O)=O)\c2csc(...
3,2-methylpropanol,1,CC(C)CO
4,spirilene,1,C1=CC=CC=C1N2C4(C(NC2)=O)CCN(CC\C=C(C3=CC=C(F)...


## ⚖️ Step 3: Bemis-Murcko Scaffold Splitting
Groups molecules based on their core cyclic skeleton and splits groups into Train (80%), Validation (10%), and Test (10%) sets, preventing structural leakages.

In [5]:
def scaffold_split(df, smiles_col='smiles', n_folds=5):
    print("  [Data Engine] Grouping molecules by Bemis-Murcko scaffolds for 5-fold CV...")
    scaffolds = defaultdict(list)
    for idx, row in df.iterrows():
        mol = Chem.MolFromSmiles(row[smiles_col])
        if mol:
            scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
            scaffolds[scaffold].append(idx)
        else:
            scaffolds[''].append(idx)
            
    groups = sorted(scaffolds.values(), key=len, reverse=True)
    folds = [[] for _ in range(n_folds)]
    fold_sizes = [0] * n_folds
    
    for group in groups:
        smallest = min(range(n_folds), key=lambda i: fold_sizes[i])
        folds[smallest].extend(group)
        fold_sizes[smallest] += len(group)
        
    return [np.array(f) for f in folds]

folds = scaffold_split(df)
for i, f in enumerate(folds):
    print(f"💾 Fold {i+1} Size: {len(f)}")


  [Data Engine] Grouping molecules by Bemis-Murcko scaffolds for 5-fold CV...
💾 Fold 1 Size: 624
💾 Fold 2 Size: 624
💾 Fold 3 Size: 624
💾 Fold 4 Size: 624
💾 Fold 5 Size: 624


## 🧬 Step 4: Molecular Feature Engineering
Extracts multiple representations from SMILES sequences:
1. **Morgan Fingerprints (ECFP4)** for RF & SVM models.
2. **RDKit 1D/2D Descriptors** for tree baselines (GBDT).
3. **Molecular Graphs** for Custom GCN and Fusion Models.

In [6]:
class MolecularPropertyEncoder:
    def __init__(self, permitted_value_sets):
        self.total_dimensions = 0
        self.property_mapping_dict = {}
        for prop_id, permitted_values in permitted_value_sets.items():
            ordered = sorted(list(permitted_values))
            mapping = dict(zip(ordered, range(self.total_dimensions, len(ordered) + self.total_dimensions)))
            self.property_mapping_dict[prop_id] = mapping
            self.total_dimensions += len(ordered)

    def transform_to_vector(self, component):
        vec = np.zeros((self.total_dimensions,))
        for prop_id, mapping in self.property_mapping_dict.items():
            val = getattr(self, prop_id)(component)
            if val in mapping:
                vec[mapping[val]] = 1.0
        return vec

class AtomicPropertyEncoder(MolecularPropertyEncoder):
    def symbol(self, a): return a.GetSymbol()
    def n_valence(self, a): return a.GetTotalValence()
    def n_hydrogens(self, a): return a.GetTotalNumHs()
    def hybridization(self, a): return a.GetHybridization().name.lower()

class BondPropertyEncoder(MolecularPropertyEncoder):
    def __init__(self, s):
        super().__init__(s)
        self.total_dimensions += 1
    def transform_to_vector(self, bond):
        vec = np.zeros((self.total_dimensions,))
        if bond is None:
            vec[-1] = 1.0
            return vec
        return super().transform_to_vector(bond)
    def bond_type(self, b): return b.GetBondType().name.lower()
    def conjugated(self, b): return b.GetIsConjugated()

atom_enc = AtomicPropertyEncoder({"symbol": {"B","Br","C","Ca","Cl","F","H","I","N","Na","O","P","S"},
    "n_valence": {0,1,2,3,4,5,6}, "n_hydrogens": {0,1,2,3,4}, "hybridization": {"s","sp","sp2","sp3"}})
bond_enc = BondPropertyEncoder({"bond_type": {"single","double","triple","aromatic"}, "conjugated": {True, False}})

def smiles_to_mol(s):
    mol = Chem.MolFromSmiles(s, sanitize=False)
    res = Chem.SanitizeMol(mol, catchErrors=True)
    if res != Chem.SanitizeFlags.SANITIZE_NONE:
        Chem.SanitizeMol(mol, sanitizeOps=Chem.SanitizeFlags.SANITIZE_ALL ^ res)
    Chem.AssignStereochemistry(mol, cleanIt=True, force=True)
    return mol

def mol_to_graph(mol):
    atom_feats, bond_feats, pairs = [], [], []
    for atom in mol.GetAtoms():
        atom_feats.append(atom_enc.transform_to_vector(atom))
        idx = atom.GetIdx()
        pairs.append([idx, idx])
        bond_feats.append(bond_enc.transform_to_vector(None))
        for nb in atom.GetNeighbors():
            nb_idx = nb.GetIdx()
            bond = mol.GetBondBetweenAtoms(idx, nb_idx)
            pairs.append([idx, nb_idx])
            bond_feats.append(bond_enc.transform_to_vector(bond))
    return np.array(atom_feats), np.array(bond_feats), np.array(pairs)

def featurize_smiles_list(smiles_list):
    atoms, bonds, conns = [], [], []
    for s in smiles_list:
        a, b, c = mol_to_graph(smiles_to_mol(s))
        atoms.append(a); bonds.append(b); conns.append(c)
    return atoms, bonds, conns

def smiles_to_ecfp4(smiles_list, n_bits=1024):
    fps = []
    for s in smiles_list:
        mol = Chem.MolFromSmiles(s)
        if mol is None:
            fps.append(np.zeros((n_bits,)))
            continue
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=n_bits)
        arr = np.zeros((0,), dtype=np.int8)
        DataStructs.ConvertToNumpyArray(fp, arr)
        fps.append(arr)
    return np.array(fps)

descriptor_names = [desc[0] for desc in Descriptors._descList]

def smiles_to_descriptors(smiles_list):
    feats_list = []
    for s in smiles_list:
        mol = Chem.MolFromSmiles(s)
        if mol is None:
            feats_list.append([0.0] * len(descriptor_names))
            continue
        feats = []
        for name in descriptor_names:
            func = getattr(Descriptors, name)
            try:
                val = func(mol)
                if np.isnan(val) or np.isinf(val):
                    val = 0.0
                feats.append(val)
            except:
                feats.append(0.0)
        feats_list.append(feats)
    return np.array(feats_list)

## ⚙️ Step 5: Preprocessing Features & Preparing DataLoaders
Scales 1D descriptors, imputes missing property values, and formats Pytorch loaders for Graph/Sequence channels.

In [7]:
# Preprocess FP & Descriptor Data Globally
smiles_all = df.smiles.tolist()
ecfp4_all = smiles_to_ecfp4(smiles_all)
descriptors_all = smiles_to_descriptors(smiles_all)

y_all = df.permeability_target.values

# Preprocess Molecular Graphs Globally
graphs_all = featurize_smiles_list(smiles_all)
atom_dim = graphs_all[0][0].shape[1]
bond_dim = graphs_all[1][0].shape[1]

# Dataset Loading & Tokenizer Setup
class FusionDataset(Dataset):
    def __init__(self, atom_feats, bond_feats, conn_indices, smiles_list, labels):
        self.atom_feats = atom_feats
        self.bond_feats = bond_feats
        self.conn_indices = conn_indices
        self.smiles_list = smiles_list
        self.labels = labels.values if hasattr(labels, 'values') else labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return (self.atom_feats[idx], self.bond_feats[idx], self.conn_indices[idx],
                self.smiles_list[idx], self.labels[idx])

def collate_fusion(batch, tokenizer):
    atoms, bonds, conns, batch_assign, smiles_batch, labels = [], [], [], [], [], []
    offset = 0
    for i, (a, b, c, smi, lbl) in enumerate(batch):
        n = a.shape[0]
        atoms.append(torch.tensor(a, dtype=torch.float32))
        bonds.append(torch.tensor(b, dtype=torch.float32))
        ct = torch.tensor(c, dtype=torch.long) + offset
        conns.append(ct)
        batch_assign.append(torch.full((n,), i, dtype=torch.long))
        smiles_batch.append(smi)
        labels.append(lbl)
        offset += n
    tok_out = tokenizer(smiles_batch, return_tensors="pt", padding=True, truncation=True, max_length=202)
    return (torch.cat(atoms), torch.cat(bonds), torch.cat(conns), torch.cat(batch_assign),
            tok_out["input_ids"], tok_out["attention_mask"],
            torch.tensor(labels, dtype=torch.float32).unsqueeze(1))

tokenizer = AutoTokenizer.from_pretrained(MOLFORMER_NAME, trust_remote_code=True)
molformer_base = AutoModel.from_pretrained(MOLFORMER_NAME, trust_remote_code=True)
molformer_base.eval()
collate_fn = lambda batch: collate_fusion(batch, tokenizer)
print("✨ Global features and tokenizer ready!")


✨ Global features and tokenizer ready!


## 🧠 Step 6: PyTorch GCN & Fusion Architecture Blocks
Defines edge message passing networks, attention pooling readouts, custom 3-layer GCN, and the dual-channel Fusion Model.

In [8]:
class AttentionReadout(nn.Module):
    def __init__(self, dim=64, heads=8, ff=512):
        super().__init__()
        self.mha = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(dim, ff), nn.ReLU(), nn.Linear(ff, dim))
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)
        
    def forward(self, feats, batch_idx):
        bs = int(batch_idx.max().item()) + 1
        parts, counts = [], []
        for i in range(bs):
            m = (batch_idx == i)
            if m.any():
                parts.append(feats[m]); counts.append(parts[-1].size(0))
        if not parts: return torch.zeros(0, feats.size(1), device=feats.device)
        mx = max(counts); d = feats.size(1)
        padded = []
        for p in parts:
            if p.size(0) < mx:
                padded.append(torch.cat([p, torch.zeros(mx - p.size(0), d, device=p.device)]))
            else:
                padded.append(p)
        x = torch.stack(padded)
        valid = (x != 0).any(dim=-1)
        att_out, _ = self.mha(x, x, x, key_padding_mask=~valid)
        x2 = self.ln1(x + att_out)
        x3 = self.ln2(x2 + self.ff(x2))
        mask = valid.unsqueeze(-1).float()
        return (x3 * mask).sum(1) / mask.sum(1).clamp(min=1)

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
        
    def forward(self, atom_feats, conn_indices):
        n_atoms = atom_feats.size(0)
        deg = torch.zeros(n_atoms, device=atom_feats.device)
        deg.scatter_add_(0, conn_indices[:, 0], torch.ones(conn_indices.size(0), device=atom_feats.device))
        deg_inv_sqrt = torch.pow(deg.clamp(min=1), -0.5)
        
        norm_feats = atom_feats * deg_inv_sqrt.unsqueeze(-1)
        gathered = norm_feats[conn_indices[:, 1]]
        
        aggregated = torch.zeros(n_atoms, norm_feats.size(1), device=atom_feats.device, dtype=atom_feats.dtype)
        idx = conn_indices[:, 0].unsqueeze(-1).expand(-1, gathered.size(1))
        aggregated.scatter_add_(0, idx, gathered)
        
        aggregated = aggregated * deg_inv_sqrt.unsqueeze(-1)
        return F.relu(self.linear(aggregated))

class GCNOnly(nn.Module):
    def __init__(self, atom_dim):
        super().__init__()
        self.conv1 = GCNLayer(atom_dim, MPNN_DIM)
        self.conv2 = GCNLayer(MPNN_DIM, MPNN_DIM)
        self.conv3 = GCNLayer(MPNN_DIM, MPNN_DIM)
        self.readout = AttentionReadout(MPNN_DIM)
        self.head = nn.Sequential(nn.Linear(MPNN_DIM, 256), nn.ReLU(), nn.Dropout(0.2),
                                  nn.Linear(256, 1), nn.Sigmoid())
    def forward(self, a, c, ba, **kw):
        h = self.conv1(a, c)
        h = self.conv2(h, c)
        h = self.conv3(h, c)
        return self.head(self.readout(h, ba))

class EdgeProcessor(nn.Module):
    def __init__(self, atom_dim, bond_dim):
        super().__init__()
        self.linear = nn.Linear(bond_dim, atom_dim)
    def forward(self, atom_f, bond_f, conn):
        processed = self.linear(bond_f)
        neighbors = atom_f[conn[:, 1]]
        merged = processed * neighbors
        n_atoms = atom_f.size(0)
        out = torch.zeros(n_atoms, merged.size(1), device=atom_f.device, dtype=atom_f.dtype)
        idx = conn[:, 0].unsqueeze(-1).expand(-1, merged.size(1))
        out.scatter_add_(0, idx, merged)
        return out

class MessagePassing(nn.Module):
    def __init__(self, atom_dim, bond_dim, hidden, steps=4):
        super().__init__()
        self.steps = steps
        self.edge = EdgeProcessor(atom_dim, bond_dim)
        self.pad_len = max(0, hidden - atom_dim)
        self.gru = nn.GRUCell(hidden, hidden)
    def forward(self, atom_f, bond_f, conn):
        state = F.pad(atom_f, (0, self.pad_len)) if self.pad_len > 0 else atom_f
        for _ in range(self.steps):
            msg = self.edge(atom_f, bond_f, conn)
            msg = F.pad(msg, (0, self.pad_len)) if self.pad_len > 0 else msg
            state = self.gru(msg, state)
        return state

class FusionModel(nn.Module):
    """MolFormer + MPNN Fusion Architecture."""
    def __init__(self, atom_dim, bond_dim, molformer, mode='concat', tuning_mode='frozen'):
        super().__init__()
        self.mode = mode
        self.tuning_mode = tuning_mode
        self.mp = MessagePassing(atom_dim, bond_dim, MPNN_DIM)
        self.readout = AttentionReadout(MPNN_DIM)
        self.molformer = molformer
        
        if tuning_mode == 'frozen':
            for p in self.molformer.parameters(): p.requires_grad = False
        elif tuning_mode == 'full':
            for p in self.molformer.parameters(): p.requires_grad = True
        elif tuning_mode == 'lora':
            from peft import LoraConfig, get_peft_model
            config = LoraConfig(
                r=8, lora_alpha=16, target_modules=["query", "key", "value"], 
                lora_dropout=0.1, bias="none"
            )
            self.molformer = get_peft_model(self.molformer, config)

        self.seq_proj = nn.Sequential(nn.Linear(MOLFORMER_DIM, MPNN_DIM), nn.LayerNorm(MPNN_DIM), nn.ReLU())

        if mode == 'concat':
            self.head = nn.Sequential(nn.Linear(MPNN_DIM*2, 256), nn.ReLU(), nn.Dropout(0.2),
                                      nn.Linear(256, 1), nn.Sigmoid())
        elif mode == 'gated':
            self.gate = nn.Linear(MPNN_DIM*2, MPNN_DIM)
            self.head = nn.Sequential(nn.Linear(MPNN_DIM, 256), nn.ReLU(), nn.Dropout(0.2),
                                      nn.Linear(256, 1), nn.Sigmoid())
        elif mode == 'cross_attention':
            self.cross_attn = nn.MultiheadAttention(MPNN_DIM, 4, batch_first=True)
            self.head = nn.Sequential(nn.Linear(MPNN_DIM, 256), nn.ReLU(), nn.Dropout(0.2),
                                      nn.Linear(256, 1), nn.Sigmoid())

    def forward(self, a, b, c, ba, ids, mask, **kw):
        g = self.readout(self.mp(a, b, c), ba)
        if self.tuning_mode == 'frozen':
            with torch.no_grad():
                out = self.molformer(input_ids=ids, attention_mask=mask)
        else:
            out = self.molformer(input_ids=ids, attention_mask=mask)
        s = out.last_hidden_state.mean(dim=1)
        s_proj = self.seq_proj(s)

        if self.mode == 'concat':
            z = torch.cat([g, s_proj], dim=-1)
        elif self.mode == 'gated':
            alpha = torch.sigmoid(self.gate(torch.cat([g, s_proj], dim=-1)))
            z = alpha * g + (1 - alpha) * s_proj
        elif self.mode == 'cross_attention':
            z, _ = self.cross_attn(g.unsqueeze(1), s_proj.unsqueeze(1), s_proj.unsqueeze(1))
            z = z.squeeze(1)
        return self.head(z)

## 📈 Step 7: Academic Visualizer Code (300 DPI)
Defines plotting libraries for creating professional academic figures.

In [9]:
def calculate_metrics(y_true, y_probs):
    y_preds = (y_probs >= 0.5).astype(int)
    precision, recall, _ = precision_recall_curve(y_true, y_probs)
    return {
        "ROC-AUC": roc_auc_score(y_true, y_probs),
        "PR-AUC": auc(recall, precision),
        "MCC": matthews_corrcoef(y_true, y_preds),
        "F1": f1_score(y_true, y_preds),
        "Accuracy": accuracy_score(y_true, y_preds)
    }

def generate_paper_plots(fold0_results):
    """Generates beautiful, publication-quality figures at 300 DPI using Fold-0 results."""
    colors = {
        "RF + ECFP4": "#E64B35",
        "SVM + ECFP4": "#4DBBD5",
        "GBDT + RDKit": "#00A087",
        "GCN": "#3C5488",
        "MolGraph-BBB": "#F39B7F"
    }

    # 1. ROC Curves Comparison
    plt.figure(figsize=(7.5, 6.8))
    ax = plt.gca()
    ax.set_facecolor("white")
    ax.grid(True, linestyle='--', alpha=0.5, color='#B0B0B0', zorder=0)
    for name, data in fold0_results.items():
        fpr, tpr, _ = roc_curve(data["labels"], data["probs"])
        auc_val = roc_auc_score(data["labels"], data["probs"])
        ax.plot(fpr, tpr, color=colors.get(name, "#7f7f7f"), lw=2.5, label=f"{name} (AUC = {auc_val:.4f})", zorder=3)
    ax.plot([0, 1], [0, 1], color='#7f7f7f', lw=1.5, linestyle='--', zorder=1)
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate (FPR)', fontweight='bold', fontsize=12, fontfamily='serif', labelpad=8)
    ax.set_ylabel('True Positive Rate (TPR)', fontweight='bold', fontsize=12, fontfamily='serif', labelpad=8)
    ax.set_title('ROC Curves Comparison (Fold 0)', fontweight='bold', fontsize=14, fontfamily='serif', pad=15)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_linewidth(1.5)
    ax.spines['left'].set_linewidth(1.5)
    ax.legend(loc="lower right", frameon=True, fontsize=11, shadow=True, facecolor='white', prop={'family': 'serif', 'size': 11})
    plt.savefig("results/plots/roc_curves_scaffold.png", dpi=300, bbox_inches='tight')
    plt.close()

    # 2. Precision-Recall Curves Comparison
    plt.figure(figsize=(7.5, 6.8))
    ax = plt.gca()
    ax.set_facecolor("white")
    ax.grid(True, linestyle='--', alpha=0.5, color='#B0B0B0', zorder=0)
    for name, data in fold0_results.items():
        precision, recall, _ = precision_recall_curve(data["labels"], data["probs"])
        pr_auc = auc(recall, precision)
        ax.plot(recall, precision, color=colors.get(name, "#7f7f7f"), lw=2.5, label=f"{name} (PR-AUC = {pr_auc:.4f})", zorder=3)
    ax.set_xlim([0.0, 1.05])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('Recall', fontweight='bold', fontsize=12, fontfamily='serif', labelpad=8)
    ax.set_ylabel('Precision', fontweight='bold', fontsize=12, fontfamily='serif', labelpad=8)
    ax.set_title('Precision-Recall Curves Comparison (Fold 0)', fontweight='bold', fontsize=14, fontfamily='serif', pad=15)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_linewidth(1.5)
    ax.spines['left'].set_linewidth(1.5)
    ax.legend(loc="lower left", frameon=True, fontsize=11, shadow=True, facecolor='white', prop={'family': 'serif', 'size': 11})
    plt.savefig("results/plots/pr_curves_scaffold.png", dpi=300, bbox_inches='tight')
    plt.close()


## 🔄 Step 8: Training & Evaluation Execution Loop
Loops through all 5 models, runs training schedules, saves predictions, and logs outputs.

In [10]:
def train_deep_model(model, loader, optimizer, criterion, model_type):
    model.train()
    total_loss, all_probs, all_labels = 0.0, [], []
    for batch in loader:
        batch = [t.to(device) for t in batch]
        a, b, c, ba, ids, mask, labels = batch
        optimizer.zero_grad()
        if model_type == 'gcn':
            out = model(a=a, c=c, ba=ba)
        else:
            out = model(a=a, b=b, c=c, ba=ba, ids=ids, mask=mask)
        loss = criterion(out, labels).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        all_probs.extend(out.detach().cpu().numpy().flatten())
        all_labels.extend(labels.cpu().numpy().flatten())
    return total_loss / len(all_labels)

@torch.no_grad()
def evaluate_deep_model(model, loader, model_type):
    model.eval()
    all_probs, all_labels = [], []
    for batch in loader:
        batch = [t.to(device) for t in batch]
        a, b, c, ba, ids, mask, labels = batch
        if model_type == 'gcn':
            out = model(a=a, c=c, ba=ba)
        else:
            out = model(a=a, b=b, c=c, ba=ba, ids=ids, mask=mask)
        all_probs.extend(out.cpu().numpy().flatten())
        all_labels.extend(labels.cpu().numpy().flatten())
    return np.array(all_probs), np.array(all_labels)

def train_and_eval_dl(model, train_loader, val_loader, test_loader, model_type, name, fold, epochs=EPOCHS):
    print(f"\n    Training {name} on Fold {fold+1}")
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
    criterion = nn.BCELoss(reduction='none')
    best_val_auc, patience_counter, best_weights = 0.0, 0, None

    for epoch in range(1, epochs + 1):
        loss = train_deep_model(model, train_loader, optimizer, criterion, model_type)
        val_probs, val_labels = evaluate_deep_model(model, val_loader, model_type)
        val_auc = roc_auc_score(val_labels, val_probs)
        
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 10:
                break

    if best_weights:
        model.load_state_dict({k: v.to(device) for k, v in best_weights.items()})
    test_probs, test_labels = evaluate_deep_model(model, test_loader, model_type)
    return test_probs, test_labels

fold_results = { "RF + ECFP4": [], "SVM + ECFP4": [], "GBDT + RDKit": [], "GCN": [], "MolGraph-BBB": [] }
fold0_raw_outputs = {}

for fold in range(5):
    print(f"\n{'='*40}\n🚀 STARTING FOLD {fold+1}/5\n{'='*40}")
    test_idx = folds[fold]
    val_idx = folds[(fold+1)%5]
    train_idx = np.concatenate([folds[i] for i in range(5) if i not in [fold, (fold+1)%5]])
    
    # --- ML Data Prep ---
    X_train_ecfp4, y_train = ecfp4_all[train_idx], y_all[train_idx]
    X_test_ecfp4, y_test = ecfp4_all[test_idx], y_all[test_idx]
    
    imputer = SimpleImputer(strategy='mean')
    scaler = StandardScaler()
    X_train_desc = scaler.fit_transform(imputer.fit_transform(descriptors_all[train_idx]))
    X_test_desc = scaler.transform(imputer.transform(descriptors_all[test_idx]))
    
    # --- DL Data Prep ---
    def get_dl_data(idx_list):
        return ([graphs_all[0][i] for i in idx_list], 
                [graphs_all[1][i] for i in idx_list],
                [graphs_all[2][i] for i in idx_list],
                [smiles_all[i] for i in idx_list],
                y_all[idx_list])
    
    train_loader = DataLoader(FusionDataset(*get_dl_data(train_idx)), BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(FusionDataset(*get_dl_data(val_idx)), BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(FusionDataset(*get_dl_data(test_idx)), BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    
    # 1. RF + ECFP4
    rf_model = RandomForestClassifier(n_estimators=100, random_state=SEED+fold)
    rf_model.fit(X_train_ecfp4, y_train)
    rf_probs = rf_model.predict_proba(X_test_ecfp4)[:, 1]
    fold_results["RF + ECFP4"].append({"probs": rf_probs, "labels": y_test})
    
    # 2. SVM + ECFP4
    svm_model = SVC(C=1.0, kernel='rbf', probability=True, random_state=SEED+fold)
    svm_model.fit(X_train_ecfp4, y_train)
    svm_probs = svm_model.predict_proba(X_test_ecfp4)[:, 1]
    fold_results["SVM + ECFP4"].append({"probs": svm_probs, "labels": y_test})
    
    # 3. GBDT + RDKit
    try:
        from xgboost import XGBClassifier
        gbdt_model = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=SEED+fold, eval_metric='logloss')
    except:
        from sklearn.ensemble import GradientBoostingClassifier
        gbdt_model = GradientBoostingClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=SEED+fold)
    gbdt_model.fit(X_train_desc, y_train)
    gbdt_probs = gbdt_model.predict_proba(X_test_desc)[:, 1]
    fold_results["GBDT + RDKit"].append({"probs": gbdt_probs, "labels": y_test})
    
    # 4. GCN
    set_seed(SEED+fold)
    gcn_model = GCNOnly(atom_dim).to(device)
    gcn_probs, gcn_labels = train_and_eval_dl(gcn_model, train_loader, val_loader, test_loader, 'gcn', 'GCN', fold)
    fold_results["GCN"].append({"probs": gcn_probs, "labels": gcn_labels})
    
    # 5. MolGraph-BBB (MolFormer + MPNN Fusion-Concat)
    set_seed(SEED+fold)
    fusion_model = FusionModel(atom_dim, bond_dim, molformer_base, mode='concat', tuning_mode='frozen').to(device)
    ours_probs, ours_labels = train_and_eval_dl(fusion_model, train_loader, val_loader, test_loader, 'fusion', 'MolGraph-BBB', fold)
    fold_results["MolGraph-BBB"].append({"probs": ours_probs, "labels": ours_labels})
    
    if fold == 0:
        torch.save(fusion_model.state_dict(), "results/best_model_MolGraph-BBB.pt")
        print("  --> Saved MolGraph-BBB Fold 0 weights for Interpretability Analysis.")
        fold0_raw_outputs = {
            "RF + ECFP4": {"probs": rf_probs, "labels": y_test},
            "SVM + ECFP4": {"probs": svm_probs, "labels": y_test},
            "GBDT + RDKit": {"probs": gbdt_probs, "labels": y_test},
            "GCN": {"probs": gcn_probs, "labels": gcn_labels},
            "MolGraph-BBB": {"probs": ours_probs, "labels": ours_labels}
        }



🚀 STARTING FOLD 1/5

    Training GCN on Fold 1

    Training MolGraph-BBB on Fold 1
  --> Saved MolGraph-BBB Fold 0 weights for Interpretability Analysis.

🚀 STARTING FOLD 2/5

    Training GCN on Fold 2

    Training MolGraph-BBB on Fold 2

🚀 STARTING FOLD 3/5

    Training GCN on Fold 3

    Training MolGraph-BBB on Fold 3

🚀 STARTING FOLD 4/5

    Training GCN on Fold 4

    Training MolGraph-BBB on Fold 4

🚀 STARTING FOLD 5/5

    Training GCN on Fold 5

    Training MolGraph-BBB on Fold 5


## 📊 Step 9: Final Metrics Analysis & Plotting
Generates comparison curves and saves the results in a publication-quality format.

In [11]:
# Compute and Print Table Summary (Mean ± SD)
metrics_summary_mean = {}
metrics_summary_std = {}

for name, folds_data in fold_results.items():
    metrics_list = {'ROC-AUC': [], 'PR-AUC': [], 'MCC': [], 'F1': [], 'Accuracy': []}
    for data in folds_data:
        m = calculate_metrics(data["labels"], data["probs"])
        for k, v in m.items():
            metrics_list[k].append(v)
    
    metrics_summary_mean[name] = {k: np.mean(v) for k, v in metrics_list.items()}
    metrics_summary_std[name] = {k: np.std(v) for k, v in metrics_list.items()}

print("\n" + "=" * 95)
print("  EXPERIMENT 2 RESULTS SUMMARY (Bemis-Murcko Scaffold 5-Fold CV)")
print("=" * 95)
print(f"  {'Model':<18s} {'ROC-AUC':>14s} {'PR-AUC':>14s} {'MCC':>14s} {'F1':>14s} {'Accuracy':>14s}")
print("-" * 95)
for name in metrics_summary_mean.keys():
    m_mean = metrics_summary_mean[name]
    m_std = metrics_summary_std[name]
    roc = f"{m_mean['ROC-AUC']:.4f}±{m_std['ROC-AUC']:.4f}"
    pr = f"{m_mean['PR-AUC']:.4f}±{m_std['PR-AUC']:.4f}"
    mcc = f"{m_mean['MCC']:.4f}±{m_std['MCC']:.4f}"
    f1 = f"{m_mean['F1']:.4f}±{m_std['F1']:.4f}"
    acc = f"{m_mean['Accuracy']:.4f}±{m_std['Accuracy']:.4f}"
    print(f"  {name:<18s} {roc:>14s} {pr:>14s} {mcc:>14s} {f1:>14s} {acc:>14s}")
print("=" * 95)

# Generate plots using Fold 0 representations
generate_paper_plots(fold0_raw_outputs)

# Bar chart for Mean Metrics
plt.figure(figsize=(10, 6.5))
ax = plt.gca()
ax.set_facecolor("white")
ax.grid(axis='y', linestyle='--', alpha=0.7, color='#B0B0B0', zorder=0)
metrics_keys = ["ROC-AUC", "PR-AUC", "MCC", "F1"]
models = list(metrics_summary_mean.keys())
x = np.arange(len(metrics_keys))
width = 0.15
colors = {"RF + ECFP4": "#E64B35", "SVM + ECFP4": "#4DBBD5", "GBDT + RDKit": "#00A087", "GCN": "#3C5488", "MolGraph-BBB": "#F39B7F"}
for i, model_name in enumerate(models):
    vals = [metrics_summary_mean[model_name][m] for m in metrics_keys]
    errs = [metrics_summary_std[model_name][m] for m in metrics_keys]
    offset = (i - len(models)/2 + 0.5) * width
    ax.bar(x + offset, vals, width, yerr=errs, capsize=4, label=model_name, 
           color=colors.get(model_name, "#7f7f7f"), edgecolor='black', linewidth=1.2, zorder=3)
ax.set_ylabel('Metric Score (Mean ± SD)', fontweight='bold', fontsize=13, fontfamily='serif', labelpad=8)
ax.set_title('Performance Metrics Comparison under 5-Fold Scaffold Split', fontweight='bold', fontsize=15, fontfamily='serif', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(metrics_keys, fontweight='bold', fontsize=12, fontfamily='serif')
ax.set_ylim([0, 1.05])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_linewidth(1.5)
ax.spines['left'].set_linewidth(1.5)
ax.legend(bbox_to_anchor=(0.5, -0.15), loc="upper center", ncol=3, frameon=False, prop={'family': 'serif', 'size': 11})
plt.tight_layout()
plt.savefig("results/plots/metrics_comparison_5fold.png", dpi=300, bbox_inches='tight')
plt.close()

# Save Results table to CSV
results_rows = []
for name in models:
    row = {"Model": name}
    for k in metrics_keys + ["Accuracy"]:
        row[k] = f"{metrics_summary_mean[name][k]:.4f}±{metrics_summary_std[name][k]:.4f}"
    results_rows.append(row)
pd.DataFrame(results_rows).to_csv("results/experiment2_5fold_results.csv", index=False)
print(f"\nResults table saved to results/experiment2_5fold_results.csv")



  EXPERIMENT 2 RESULTS SUMMARY (Bemis-Murcko Scaffold 5-Fold CV)
  Model                     ROC-AUC         PR-AUC            MCC             F1       Accuracy
-----------------------------------------------------------------------------------------------
  RF + ECFP4          0.8735±0.0362  0.8672±0.0340  0.5406±0.1103  0.7899±0.0544  0.7417±0.0753
  SVM + ECFP4         0.8838±0.0360  0.8704±0.0370  0.5477±0.1102  0.7937±0.0478  0.7519±0.0677
  GBDT + RDKit        0.8923±0.0476  0.8811±0.0499  0.5814±0.1018  0.8065±0.0473  0.7763±0.0613
  GCN                 0.8265±0.0391  0.8054±0.0317  0.5105±0.0904  0.7614±0.0431  0.7506±0.0550
  MolGraph-BBB        0.9386±0.0147  0.9489±0.0074  0.6860±0.0602  0.8485±0.0208  0.8433±0.0322

Results table saved to results/experiment2_5fold_results.csv
